# 06 · Mapa interativo com Folium

> **Atenção:** execute todas as células (Kernel → Restart & Run All) e salve o notebook **com as saídas** antes de enviar ao GitHub. O curso exige células de código e de saída visíveis.

In [ ]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from math import sin, cos, sqrt, atan2, radians

sites = pd.DataFrame({
    'Launch Site': ['CCAFS LC-40', 'CCAFS SLC-40', 'KSC LC-39A', 'VAFB SLC-4E'],
    'Lat': [28.562302, 28.563197, 28.573255, 34.632834],
    'Long': [-80.577356, -80.576820, -80.646895, -120.610746]})
sites

### Mapa global com os locais de lançamento

In [ ]:
m = folium.Map(location=[29.559684888503615, -95.0830971930759], zoom_start=4)
for _, r in sites.iterrows():
    folium.Circle([r.Lat, r.Long], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(m)
    folium.Marker([r.Lat, r.Long], icon=folium.DivIcon(html=f'<div style="font-size:12;color:#d35400"><b>{r["Launch Site"]}</b></div>')).add_to(m)
m

### Resultados coloridos (verde = sucesso, vermelho = falha)
Use `spacex_launch_geo.csv` (colunas `Launch Site`, `Lat`, `Long`, `class`).

In [ ]:
geo = pd.read_csv('../data/spacex_launch_geo.csv')
geo['marker_color'] = geo['class'].apply(lambda c: 'green' if c == 1 else 'red')
cluster = MarkerCluster().add_to(m)
for _, r in geo.iterrows():
    folium.Marker([r['Lat'], r['Long']], icon=folium.Icon(color='white', icon_color=r['marker_color'])).add_to(cluster)
m

### Distâncias até as proximidades (Haversine)

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

MousePosition().add_to(m)   # use o mouse para ler as coordenadas da costa, ferrovia, rodovia e cidade
# exemplo: substitua as coordenadas pelas que você capturou no mapa
coast_lat, coast_lon = 28.56367, -80.57163
d = calculate_distance(28.563197, -80.576820, coast_lat, coast_lon)
folium.PolyLine([[28.563197, -80.576820], [coast_lat, coast_lon]], weight=1).add_to(m)
folium.Marker([coast_lat, coast_lon], icon=folium.DivIcon(html=f'<div style="font-size:12">{d:.2f} KM</div>')).add_to(m)
m